# Advanced Problems with Solutions: Dictionary Views (`keys`, `values`, `items`)

This notebook develops deep practical understanding of Python dictionary view objects.

## Topics covered

- Dynamic/live behavior of dictionary views
- `dict_keys`, `dict_values`, and `dict_items`
- Safe and unsafe mutation during iteration
- Snapshot strategies
- Set-like behavior of key and item views
- Value-view limitations
- Iterator invalidation
- Updating values safely
- Filtering and transforming dictionaries
- `popitem()` processing patterns
- Reverse iteration
- View `.mapping`
- Membership and equality semantics
- Hashability edge cases
- Performance and benchmarking
- Production-style helper functions
- Debugging subtle mutation bugs

The problems are intentionally more advanced than basic dictionary exercises. Each problem is followed by a complete solution and explanation.

> Best practice: run the notebook from top to bottom so all assertions and demonstrations execute in order.


In [1]:
from __future__ import annotations

from collections.abc import Mapping
from timeit import timeit
from types import MappingProxyType
from typing import Any, Callable, Iterable

print("Notebook setup complete.")


Notebook setup complete.


## Quick Reference

A dictionary's views are **live**:

```python
d = {"a": 1}
keys = d.keys()
d["b"] = 2
# keys now sees both "a" and "b"
```

Key rules:

1. `d.keys()` is a dynamic view over keys.
2. `d.values()` is a dynamic view over values.
3. `d.items()` is a dynamic view over `(key, value)` pairs.
4. Changing an existing value while iterating is normally safe because dictionary size does not change.
5. Adding or removing keys during normal dictionary iteration can raise `RuntimeError`.
6. If structural mutation is required, iterate over a **snapshot** such as `list(d)` or `tuple(d.items())`.
7. `dict_keys` supports set-like operations.
8. `dict_items` supports many set-like operations when the involved items are hashable.
9. `dict_values` is **not** set-like.
10. Prefer `for key, value in d.items()` when both are needed.


# Problem 1 — Prove That Views Are Live

Create a dictionary and store all three view objects. Mutate the dictionary in several ways:

- add a key,
- change an existing value,
- delete a key.

Without recreating the views, prove that all three views reflect the current dictionary state.

Also verify that the *view objects themselves* remain the same objects.


In [2]:
# Solution 1

d = {"alpha": 10, "beta": 20}

keys_view = d.keys()
values_view = d.values()
items_view = d.items()

ids_before = tuple(map(id, (keys_view, values_view, items_view)))

print("Initially:")
print(keys_view)
print(values_view)
print(items_view)

d["gamma"] = 30
d["alpha"] = 999
del d["beta"]

ids_after = tuple(map(id, (keys_view, values_view, items_view)))

print("\nAfter mutations:")
print(keys_view)
print(values_view)
print(items_view)

assert ids_before == ids_after
assert set(keys_view) == {"alpha", "gamma"}
assert d["alpha"] == 999
assert ("alpha", 999) in items_view

print("\nSame view objects:", ids_before == ids_after)


Initially:
dict_keys(['alpha', 'beta'])
dict_values([10, 20])
dict_items([('alpha', 10), ('beta', 20)])

After mutations:
dict_keys(['alpha', 'gamma'])
dict_values([999, 30])
dict_items([('alpha', 999), ('gamma', 30)])

Same view objects: True


### Why this works

A dictionary view does not store an independent copy of the dictionary's contents. It references the underlying dictionary and computes its visible contents from the dictionary's current state.

The fact that `id(keys_view)` stays unchanged does **not** imply that its contents are frozen.


# Problem 2 — Snapshot vs Live View

You need two objects:

- one that always reflects the dictionary's current keys,
- one that preserves the keys exactly as they existed at a particular moment.

Implement both and demonstrate the difference after three mutations.


In [3]:
# Solution 2

d = {"a": 1, "b": 2, "c": 3}

live_keys = d.keys()
snapshot_keys = tuple(d)

d["d"] = 4
del d["a"]
d["e"] = 5

print("Live keys:    ", tuple(live_keys))
print("Snapshot keys:", snapshot_keys)

assert tuple(live_keys) == ("b", "c", "d", "e")
assert snapshot_keys == ("a", "b", "c")


Live keys:     ('b', 'c', 'd', 'e')
Snapshot keys: ('a', 'b', 'c')


### Best practice

Use a view when you intentionally want a live window.

Use a snapshot when:

- you need stable iteration while structurally mutating the dictionary,
- you need historical state,
- you need deterministic processing independent of later mutations.


# Problem 3 — Diagnose an Iterator-Invalidation Bug

The following code is intended to delete all entries whose values are negative:

```python
scores = {"a": 10, "b": -4, "c": 8, "d": -1}

for key, value in scores.items():
    if value < 0:
        del scores[key]
```

Explain the bug and implement **three safe fixes**:

1. iterate over a list snapshot,
2. collect keys first and delete later,
3. rebuild the dictionary with a comprehension.


In [4]:
# Solution 3

original = {"a": 10, "b": -4, "c": 8, "d": -1}

# Fix 1: snapshot
scores1 = original.copy()
for key, value in list(scores1.items()):
    if value < 0:
        del scores1[key]

# Fix 2: two-phase delete
scores2 = original.copy()
to_delete = [key for key, value in scores2.items() if value < 0]
for key in to_delete:
    del scores2[key]

# Fix 3: rebuild
scores3 = {key: value for key, value in original.items() if value >= 0}

expected = {"a": 10, "c": 8}

assert scores1 == expected
assert scores2 == expected
assert scores3 == expected

print(scores1)
print(scores2)
print(scores3)


{'a': 10, 'c': 8}
{'a': 10, 'c': 8}
{'a': 10, 'c': 8}


### Choosing among the fixes

- **Snapshot mutation** is direct and readable.
- **Two-phase deletion** separates discovery from mutation and is often easiest to debug.
- **Dictionary comprehension** is often best when conceptually creating a filtered dictionary.

For production code, prefer the approach that best expresses intent rather than minimizing line count.


# Problem 4 — Safe Value Mutation During Iteration

Normalize the values in this dictionary so every value is converted to a percentage of the total:

```python
weights = {"cpu": 2, "memory": 3, "disk": 5}
```

Perform the update **in place** while iterating over the dictionary.


In [5]:
# Solution 4

weights = {"cpu": 2, "memory": 3, "disk": 5}
total = sum(weights.values())

for key in weights:
    weights[key] = weights[key] / total

print(weights)

assert abs(sum(weights.values()) - 1.0) < 1e-12
assert weights == {"cpu": 0.2, "memory": 0.3, "disk": 0.5}


{'cpu': 0.2, 'memory': 0.3, 'disk': 0.5}


### Why it is safe

The set of keys is unchanged. Only values associated with existing keys are replaced.

This does not structurally resize the dictionary.


# Problem 5 — Use `.items()` Instead of Repeated Lookups

Write two functions that sum `key * value` for an integer dictionary:

- `sum_lookup_style`: iterate over keys and repeatedly use `d[key]`.
- `sum_items_style`: iterate directly over `d.items()`.

Verify correctness and benchmark both approaches.


In [6]:
# Solution 5

def sum_lookup_style(d: dict[int, int]) -> int:
    total = 0
    for key in d:
        total += key * d[key]
    return total


def sum_items_style(d: dict[int, int]) -> int:
    total = 0
    for key, value in d.items():
        total += key * value
    return total


sample = {i: i % 17 for i in range(20_000)}

assert sum_lookup_style(sample) == sum_items_style(sample)

lookup_time = timeit(lambda: sum_lookup_style(sample), number=200)
items_time = timeit(lambda: sum_items_style(sample), number=200)

print(f"Lookup style: {lookup_time:.4f}s")
print(f"Items style:  {items_time:.4f}s")
print(f"Ratio lookup/items: {lookup_time / items_time:.2f}x")


Lookup style: 0.3664s
Items style:  0.3148s
Ratio lookup/items: 1.16x


### Benchmarking note

Exact timings vary by Python version, CPU, operating system, and runtime state.

The important design principle is that `items()` directly supplies both key and value and avoids an explicit second dictionary lookup in your Python code.


# Problem 6 — Set Algebra with `dict_keys`

Two services expose feature flags:

```python
service_a = {"search": True, "upload": True, "share": False, "export": True}
service_b = {"search": True, "share": True, "billing": True}
```

Using key views directly, compute:

- features supported by both,
- features only in A,
- features only in B,
- all distinct feature names,
- features supported by exactly one service.


In [7]:
# Solution 6

service_a = {"search": True, "upload": True, "share": False, "export": True}
service_b = {"search": True, "share": True, "billing": True}

a_keys = service_a.keys()
b_keys = service_b.keys()

both = a_keys & b_keys
only_a = a_keys - b_keys
only_b = b_keys - a_keys
all_features = a_keys | b_keys
exactly_one = a_keys ^ b_keys

print("Both:", both)
print("Only A:", only_a)
print("Only B:", only_b)
print("All:", all_features)
print("Exactly one:", exactly_one)

assert both == {"search", "share"}
assert only_a == {"upload", "export"}
assert only_b == {"billing"}
assert exactly_one == {"upload", "export", "billing"}


Both: {'search', 'share'}
Only A: {'upload', 'export'}
Only B: {'billing'}
All: {'search', 'share', 'billing', 'upload', 'export'}
Exactly one: {'billing', 'upload', 'export'}


### Best practice

Do not convert key views to sets unless you specifically need an independent set object.

Operations such as `d1.keys() & d2.keys()` are already supported and clearly communicate that you are comparing dictionary key domains.


# Problem 7 — Detect Changed, Added, and Removed Configuration Keys

Given an old and new configuration, classify keys into:

- added,
- removed,
- unchanged values,
- changed values.

Use dictionary views where appropriate.


In [8]:
# Solution 7

old = {
    "host": "db.internal",
    "port": 5432,
    "timeout": 10,
    "retries": 3,
}

new = {
    "host": "db.internal",
    "port": 6432,
    "timeout": 10,
    "ssl": True,
}

old_keys = old.keys()
new_keys = new.keys()

added = new_keys - old_keys
removed = old_keys - new_keys
common = old_keys & new_keys

unchanged = {k for k in common if old[k] == new[k]}
changed = {k for k in common if old[k] != new[k]}

print("Added:", added)
print("Removed:", removed)
print("Unchanged:", unchanged)
print("Changed:", changed)

assert added == {"ssl"}
assert removed == {"retries"}
assert unchanged == {"host", "timeout"}
assert changed == {"port"}


Added: {'ssl'}
Removed: {'retries'}
Unchanged: {'timeout', 'host'}
Changed: {'port'}


# Problem 8 — Item-View Set Operations

Suppose two dictionaries represent exact user-role assignments:

```python
before = {"alice": "admin", "bob": "viewer", "carol": "editor"}
after  = {"alice": "admin", "bob": "editor", "dave": "viewer"}
```

Use item views to identify exact `(user, role)` assignments that:

- remained unchanged,
- disappeared,
- appeared.


In [9]:
# Solution 8

before = {"alice": "admin", "bob": "viewer", "carol": "editor"}
after = {"alice": "admin", "bob": "editor", "dave": "viewer"}

unchanged_pairs = before.items() & after.items()
removed_pairs = before.items() - after.items()
added_pairs = after.items() - before.items()

print("Unchanged:", unchanged_pairs)
print("Removed:", removed_pairs)
print("Added:", added_pairs)

assert unchanged_pairs == {("alice", "admin")}
assert removed_pairs == {("bob", "viewer"), ("carol", "editor")}
assert added_pairs == {("bob", "editor"), ("dave", "viewer")}


Unchanged: {('alice', 'admin')}
Removed: {('carol', 'editor'), ('bob', 'viewer')}
Added: {('bob', 'editor'), ('dave', 'viewer')}


### Important caveat: hashability

Set-like item-view operations require the relevant `(key, value)` pairs to be hashable.

Keys are already required to be hashable, but dictionary values may be unhashable.

For example, a value that is a list can make a set operation fail.


In [10]:
# Hashability edge case

left = {"a": [1, 2]}
right = {"a": [1, 2]}

try:
    result = left.items() & right.items()
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: unhashable type: 'list'


# Problem 9 — Why `dict_values` Is Not a Set

Investigate the following questions:

1. Does `d.values() == d.values()` evaluate to `True`?
2. Can you use `&` between two value views?
3. What should you do if you genuinely need set semantics for values?


In [11]:
# Solution 9

d = {"a": 1, "b": 2, "c": 2}

v1 = d.values()
v2 = d.values()

print("v1 == v1:", v1 == v1)
print("v1 == v2:", v1 == v2)

try:
    print(v1 & v2)
except TypeError as exc:
    print("Set operation failed:", exc)

unique_values = set(d.values())
print("Unique values:", unique_values)

assert unique_values == {1, 2}


v1 == v1: True
v1 == v2: False
Set operation failed: unsupported operand type(s) for &: 'dict_values' and 'dict_values'
Unique values: {1, 2}


### Key idea

`dict_values` deliberately does not behave like a mathematical set because:

- values can be duplicated,
- values can be unhashable,
- order and multiplicity may matter to the application.

If set semantics are required and values are hashable, explicitly use `set(d.values())`.


# Problem 10 — Live View in a Long-Lived Object

Create a class `RegistryInspector` that receives a dictionary and stores a key view once during initialization.

The class must provide:

- `visible_names()` → current names as a tuple,
- `count()` → current number of keys.

Show that no view recreation is necessary after the registry changes.


In [12]:
# Solution 10

class RegistryInspector:
    def __init__(self, registry: dict[str, Any]) -> None:
        self._keys = registry.keys()

    def visible_names(self) -> tuple[str, ...]:
        return tuple(self._keys)

    def count(self) -> int:
        return len(self._keys)


registry = {"parser": object(), "renderer": object()}
inspector = RegistryInspector(registry)

print(inspector.visible_names())

registry["validator"] = object()
del registry["parser"]

print(inspector.visible_names())
print(inspector.count())

assert inspector.visible_names() == ("renderer", "validator")
assert inspector.count() == 2


('parser', 'renderer')
('renderer', 'validator')
2


# Problem 11 — Structural Mutation Hidden Inside a Function

A helper function modifies a dictionary, but the caller is iterating over that dictionary:

```python
def deactivate_if_needed(users, name):
    if users[name]["active"] is False:
        del users[name]

for name in users:
    deactivate_if_needed(users, name)
```

Refactor this design so mutation is safe and responsibilities are clearer.


In [13]:
# Solution 11

users = {
    "alice": {"active": True},
    "bob": {"active": False},
    "carol": {"active": True},
    "dave": {"active": False},
}

def names_to_remove(users: Mapping[str, Mapping[str, Any]]) -> list[str]:
    return [
        name
        for name, record in users.items()
        if record.get("active") is False
    ]


for name in names_to_remove(users):
    del users[name]

print(users)

assert set(users) == {"alice", "carol"}


{'alice': {'active': True}, 'carol': {'active': True}}


### Design lesson

Hidden structural mutation inside a helper can make iterator-invalidating bugs difficult to diagnose.

A strong pattern is:

1. inspect,
2. decide,
3. mutate.

This separates read-only traversal from structural changes.


# Problem 12 — Process and Empty a Dictionary with `popitem()`

You have a work queue stored as a dictionary. Process every task and remove it from the dictionary.

Requirements:

- do not create a list snapshot,
- do not mutate while iterating over a dictionary view,
- leave the dictionary empty.


In [14]:
# Solution 12

tasks = {
    "task-1": 5,
    "task-2": 8,
    "task-3": 13,
    "task-4": 21,
}

processed = []

while tasks:
    task_id, cost = tasks.popitem()
    processed.append((task_id, cost, cost * 2))

print("Processed:")
for row in processed:
    print(row)

print("Remaining:", tasks)

assert tasks == {}
assert len(processed) == 4


Processed:
('task-4', 21, 42)
('task-3', 13, 26)
('task-2', 8, 16)
('task-1', 5, 10)
Remaining: {}


### When `popitem()` is a good fit

Use this pattern when the dictionary itself is being consumed.

Modern Python dictionaries preserve insertion order, and `popitem()` removes the most recently inserted pair (LIFO behavior).


# Problem 13 — Reverse Iteration over Views

Given:

```python
d = {"first": 1, "second": 2, "third": 3}
```

Iterate over:

- keys in reverse insertion order,
- items in reverse insertion order,
- values in reverse insertion order.


In [15]:
# Solution 13

d = {"first": 1, "second": 2, "third": 3}

reverse_keys = list(reversed(d.keys()))
reverse_items = list(reversed(d.items()))
reverse_values = list(reversed(d.values()))

print(reverse_keys)
print(reverse_items)
print(reverse_values)

assert reverse_keys == ["third", "second", "first"]
assert reverse_items == [("third", 3), ("second", 2), ("first", 1)]
assert reverse_values == [3, 2, 1]


['third', 'second', 'first']
[('third', 3), ('second', 2), ('first', 1)]
[3, 2, 1]


# Problem 14 — Read-Only Access Through a View's `.mapping`

Modern Python dictionary views expose a `.mapping` attribute that returns a read-only mapping proxy.

Create a keys view, obtain its mapping proxy, and prove:

1. the proxy reflects later updates,
2. assignment through the proxy is forbidden.


In [16]:
# Solution 14

settings = {"mode": "dev", "debug": True}
keys = settings.keys()
readonly = keys.mapping

print("Initial proxy:", readonly)

settings["timeout"] = 30
print("After update:", readonly)

assert readonly["timeout"] == 30

try:
    readonly["mode"] = "prod"
except TypeError as exc:
    print("Expected TypeError:", exc)


Initial proxy: {'mode': 'dev', 'debug': True}
After update: {'mode': 'dev', 'debug': True, 'timeout': 30}
Expected TypeError: 'mappingproxy' object does not support item assignment


### Why this is useful

A mapping proxy is useful when you want to expose current mapping data to another part of a program without giving that consumer direct write access through the proxy.


# Problem 15 — Implement a Safe In-Place Value Transformer

Write a reusable function:

```python
transform_values_in_place(d, transform)
```

It must replace every value with `transform(key, value)` while preserving the key set.

Then use it to transform prices into tax-inclusive prices.


In [17]:
# Solution 15

def transform_values_in_place(
    d: dict[Any, Any],
    transform: Callable[[Any, Any], Any],
) -> None:
    for key, value in d.items():
        d[key] = transform(key, value)


prices = {
    "book": 20.00,
    "mouse": 35.50,
    "monitor": 210.00,
}

tax_rate = 0.20

transform_values_in_place(
    prices,
    lambda key, value: round(value * (1 + tax_rate), 2),
)

print(prices)

assert prices == {
    "book": 24.0,
    "mouse": 42.6,
    "monitor": 252.0,
}


{'book': 24.0, 'mouse': 42.6, 'monitor': 252.0}


# Problem 16 — Conditional In-Place Mutation Without Resizing

Suppose inventory counts must never be negative.

For every existing key:

- negative values become `0`,
- values greater than `100` become `100`,
- all other values stay unchanged.

Perform the operation safely in place.


In [18]:
# Solution 16

inventory = {
    "A": -5,
    "B": 30,
    "C": 170,
    "D": 0,
}

for sku, count in inventory.items():
    inventory[sku] = min(100, max(0, count))

print(inventory)

assert inventory == {
    "A": 0,
    "B": 30,
    "C": 100,
    "D": 0,
}


{'A': 0, 'B': 30, 'C': 100, 'D': 0}


# Problem 17 — A Subtle Mutation: Delete One Key and Add Another

Consider this pattern:

```python
for key in d:
    del d[key]
    d[new_key] = value
```

The dictionary size may return to its previous size after each loop body.

Is this a safe strategy?

Demonstrate why code should **not** rely on balancing insertions and deletions during iteration, then implement a safe rename operation using a snapshot.


In [19]:
# Solution 17

def rename_keys_safely(
    d: dict[str, Any],
    renamer: Callable[[str], str],
) -> None:
    # Snapshot the original key-value pairs.
    original_items = list(d.items())

    d.clear()

    for old_key, value in original_items:
        new_key = renamer(old_key)

        if new_key in d:
            raise ValueError(f"rename collision: {old_key!r} -> {new_key!r}")

        d[new_key] = value


data = {"first_name": "Ada", "last_name": "Lovelace"}

rename_keys_safely(data, str.upper)

print(data)

assert data == {
    "FIRST_NAME": "Ada",
    "LAST_NAME": "Lovelace",
}


{'FIRST_NAME': 'Ada', 'LAST_NAME': 'Lovelace'}


### Best practice

Do not reason only in terms of final dictionary size.

Structural changes can invalidate iteration assumptions even if an insertion and deletion happen to balance numerically. If keys are being renamed, deleted, or inserted, iterate over a stable snapshot or build a new dictionary.


# Problem 18 — Collision-Safe Key Renaming

Extend the previous idea.

Rename keys by lowercasing them:

```python
{"User": 1, "USER": 2}
```

This creates a collision.

Write a function that detects collisions **before committing any mutation**, so the original dictionary remains unchanged if renaming is invalid.


In [20]:
# Solution 18

def renamed_copy(
    d: Mapping[str, Any],
    renamer: Callable[[str], str],
) -> dict[str, Any]:
    result: dict[str, Any] = {}

    for old_key, value in d.items():
        new_key = renamer(old_key)

        if new_key in result:
            raise ValueError(
                f"collision while renaming {old_key!r} to {new_key!r}"
            )

        result[new_key] = value

    return result


original = {"User": 1, "USER": 2}

try:
    transformed = renamed_copy(original, str.lower)
except ValueError as exc:
    print("Detected:", exc)

print("Original remains unchanged:", original)

assert original == {"User": 1, "USER": 2}


Detected: collision while renaming 'USER' to 'user'
Original remains unchanged: {'User': 1, 'USER': 2}


# Problem 19 — Membership Semantics

For:

```python
d = {"x": 10, "y": 20}
```

Predict and verify the result of:

- `"x" in d`
- `"x" in d.keys()`
- `10 in d.values()`
- `("x", 10) in d.items()`
- `("x", 999) in d.items()`

Then explain which form is normally preferred for key membership.


In [21]:
# Solution 19

d = {"x": 10, "y": 20}

checks = {
    '"x" in d': "x" in d,
    '"x" in d.keys()': "x" in d.keys(),
    '10 in d.values()': 10 in d.values(),
    '("x", 10) in d.items()': ("x", 10) in d.items(),
    '("x", 999) in d.items()': ("x", 999) in d.items(),
}

for expression, result in checks.items():
    print(f"{expression:<28} -> {result}")

assert checks['"x" in d'] is True
assert checks['"x" in d.keys()'] is True
assert checks['10 in d.values()'] is True
assert checks['("x", 10) in d.items()'] is True
assert checks['("x", 999) in d.items()'] is False


"x" in d                     -> True
"x" in d.keys()              -> True
10 in d.values()             -> True
("x", 10) in d.items()       -> True
("x", 999) in d.items()      -> False


### Best practice

For key membership, prefer:

```python
if key in d:
    ...
```

It is shorter and idiomatic.

Use `.keys()` explicitly when you need the view itself, especially for set-like operations.


# Problem 20 — Determine Whether Two Dictionaries Have the Same Key Domain

Write a function `same_keys(a, b)` that compares only dictionary key sets, ignoring values.

Then write `same_items(a, b)` that checks exact `(key, value)` equality using views when possible.


In [22]:
# Solution 20

def same_keys(a: Mapping[Any, Any], b: Mapping[Any, Any]) -> bool:
    return a.keys() == b.keys()


def same_items(a: Mapping[Any, Any], b: Mapping[Any, Any]) -> bool:
    return a.items() == b.items()


a = {"x": 1, "y": 2}
b = {"x": 100, "y": 200}
c = {"x": 1, "z": 2}
d = {"y": 2, "x": 1}

assert same_keys(a, b) is True
assert same_keys(a, c) is False
assert same_items(a, b) is False
assert same_items(a, d) is True

print("All equality tests passed.")


All equality tests passed.


# Problem 21 — Partition a Dictionary Without Unsafe Mutation

Partition a dictionary of jobs into:

- completed,
- pending.

Then remove all completed jobs from the original dictionary safely.

Return the removed jobs as a new dictionary.


In [23]:
# Solution 21

jobs = {
    "compile": "done",
    "test": "pending",
    "package": "done",
    "deploy": "pending",
}

completed_keys = [
    job
    for job, status in jobs.items()
    if status == "done"
]

completed = {
    job: jobs.pop(job)
    for job in completed_keys
}

print("Completed:", completed)
print("Remaining:", jobs)

assert completed == {
    "compile": "done",
    "package": "done",
}

assert jobs == {
    "test": "pending",
    "deploy": "pending",
}


Completed: {'compile': 'done', 'package': 'done'}
Remaining: {'test': 'pending', 'deploy': 'pending'}


# Problem 22 — Efficient Dictionary Diff Function

Implement:

```python
diff_mappings(old, new)
```

Return a dictionary with four entries:

- `"added"` → `{key: new_value}`
- `"removed"` → `{key: old_value}`
- `"changed"` → `{key: (old_value, new_value)}`
- `"unchanged"` → `{key: value}`

Use key-view set operations to avoid unnecessary scanning.


In [24]:
# Solution 22

def diff_mappings(
    old: Mapping[Any, Any],
    new: Mapping[Any, Any],
) -> dict[str, dict[Any, Any]]:
    old_keys = old.keys()
    new_keys = new.keys()

    added_keys = new_keys - old_keys
    removed_keys = old_keys - new_keys
    common_keys = old_keys & new_keys

    added = {k: new[k] for k in added_keys}
    removed = {k: old[k] for k in removed_keys}

    changed = {
        k: (old[k], new[k])
        for k in common_keys
        if old[k] != new[k]
    }

    unchanged = {
        k: old[k]
        for k in common_keys
        if old[k] == new[k]
    }

    return {
        "added": added,
        "removed": removed,
        "changed": changed,
        "unchanged": unchanged,
    }


old = {"a": 1, "b": 2, "c": 3, "e": 5}
new = {"a": 1, "b": 20, "d": 4, "e": 5}

diff = diff_mappings(old, new)
print(diff)

assert diff["added"] == {"d": 4}
assert diff["removed"] == {"c": 3}
assert diff["changed"] == {"b": (2, 20)}
assert diff["unchanged"] == {"a": 1, "e": 5}


{'added': {'d': 4}, 'removed': {'c': 3}, 'changed': {'b': (2, 20)}, 'unchanged': {'e': 5, 'a': 1}}


# Problem 23 — Avoid Accidentally Retaining a Large Dictionary

A view keeps a reference to its underlying dictionary.

Demonstrate conceptually why storing a view can keep the dictionary alive, and show how to take an independent snapshot when that behavior is not wanted.

This example uses `sys.getrefcount` only as an observational tool; exact counts are implementation-dependent.


In [25]:
# Solution 23

import sys

big_mapping = {i: i * i for i in range(10_000)}

before = sys.getrefcount(big_mapping)
view = big_mapping.keys()
after_view = sys.getrefcount(big_mapping)

snapshot = tuple(view)
after_snapshot = sys.getrefcount(big_mapping)

print("Before view:   ", before)
print("After view:    ", after_view)
print("After snapshot:", after_snapshot)

# The view references the dictionary.
# The tuple snapshot contains keys but does not need the dictionary
# to remain attached to it.

assert len(snapshot) == 10_000


Before view:    2
After view:     3
After snapshot: 3


### Engineering implication

Long-lived views are useful, but they also extend the lifetime of their underlying dictionary.

If you only need historical data, store a snapshot instead.


# Problem 24 — Compare Memory Intent: View vs Snapshot

Use `sys.getsizeof` to compare the shallow size of:

- a key view,
- a list snapshot of the same keys,
- a tuple snapshot of the same keys.

Do not interpret `getsizeof` as total transitive memory usage; use it only to illustrate the different storage strategies.


In [26]:
# Solution 24

import sys

d = {i: i for i in range(50_000)}

keys_view = d.keys()
keys_list = list(d)
keys_tuple = tuple(d)

print("keys view shallow bytes:", sys.getsizeof(keys_view))
print("list snapshot shallow bytes:", sys.getsizeof(keys_list))
print("tuple snapshot shallow bytes:", sys.getsizeof(keys_tuple))

assert len(keys_view) == len(keys_list) == len(keys_tuple) == 50_000


keys view shallow bytes: 40
list snapshot shallow bytes: 400056
tuple snapshot shallow bytes: 400040


# Problem 25 — Build a Stable Batch While the Original Dictionary May Change Later

A batch processor needs the exact `(job_id, payload)` pairs that existed at batch-start time.

After the batch is created, the original dictionary may be changed by unrelated code.

Implement a stable batch and prove it does not change.


In [27]:
# Solution 25

jobs = {
    "j1": {"priority": 1},
    "j2": {"priority": 2},
}

# This snapshots the key-value references.
batch = tuple(jobs.items())

jobs["j3"] = {"priority": 3}
del jobs["j1"]

print("Current jobs:", jobs)
print("Stable batch:", batch)

assert [job_id for job_id, _ in batch] == ["j1", "j2"]


Current jobs: {'j2': {'priority': 2}, 'j3': {'priority': 3}}
Stable batch: (('j1', {'priority': 1}), ('j2', {'priority': 2}))


### Important shallow-copy nuance

`tuple(d.items())` snapshots the key-value **references**, not recursively copied nested objects.

If mutable nested values must also be isolated, use an appropriate deep-copy or serialization strategy.


# Problem 26 — Demonstrate the Shallow Snapshot Trap

Create a dictionary whose values are mutable lists.

Take `tuple(d.items())`, mutate one of the lists through the original dictionary, and show that the snapshot still observes the mutated list.

Then create a deep-copied snapshot that does not.


In [28]:
# Solution 26

from copy import deepcopy

d = {
    "a": [1, 2],
    "b": [3, 4],
}

shallow_snapshot = tuple(d.items())
deep_snapshot = deepcopy(tuple(d.items()))

d["a"].append(999)

print("Dictionary:", d)
print("Shallow snapshot:", shallow_snapshot)
print("Deep snapshot:", deep_snapshot)

assert shallow_snapshot[0][1] == [1, 2, 999]
assert deep_snapshot[0][1] == [1, 2]


Dictionary: {'a': [1, 2, 999], 'b': [3, 4]}
Shallow snapshot: (('a', [1, 2, 999]), ('b', [3, 4]))
Deep snapshot: (('a', [1, 2]), ('b', [3, 4]))


# Problem 27 — Safely Move Entries Between Dictionaries

Move every entry with an even value from `source` to `target`.

The move must:

- remove matching entries from `source`,
- add them to `target`,
- avoid mutation-during-iteration errors.


In [29]:
# Solution 27

source = {
    "a": 1,
    "b": 2,
    "c": 3,
    "d": 4,
    "e": 5,
    "f": 6,
}

target = {}

keys_to_move = [
    key
    for key, value in source.items()
    if value % 2 == 0
]

for key in keys_to_move:
    target[key] = source.pop(key)

print("Source:", source)
print("Target:", target)

assert source == {"a": 1, "c": 3, "e": 5}
assert target == {"b": 2, "d": 4, "f": 6}


Source: {'a': 1, 'c': 3, 'e': 5}
Target: {'b': 2, 'd': 4, 'f': 6}


# Problem 28 — Merge Only Missing Keys Using Key Views

Merge values from `defaults` into `config`, but only for keys that do not already exist in `config`.

Use a key-view difference to determine the missing domain.


In [30]:
# Solution 28

defaults = {
    "host": "localhost",
    "port": 8000,
    "debug": False,
    "workers": 4,
}

config = {
    "host": "api.internal",
    "debug": True,
}

missing = defaults.keys() - config.keys()

for key in missing:
    config[key] = defaults[key]

print(config)

assert config == {
    "host": "api.internal",
    "debug": True,
    "port": 8000,
    "workers": 4,
}


{'host': 'api.internal', 'debug': True, 'workers': 4, 'port': 8000}


### Alternative production solution

For this exact use case, `dict.setdefault` or dictionary merge patterns may be even more direct.

The key-view version is useful when the *set of missing keys itself* is meaningful and may be inspected, logged, validated, or reused.


# Problem 29 — Validate an Exact Schema with Key Views

A record must contain exactly these keys:

```python
required = {"id", "name", "email"}
```

Write a validator that returns:

- whether the schema is exact,
- missing keys,
- unexpected keys.


In [31]:
# Solution 29

def validate_exact_keys(
    record: Mapping[str, Any],
    required: set[str],
) -> tuple[bool, set[str], set[str]]:
    actual = record.keys()

    missing = required - actual
    unexpected = actual - required

    return not missing and not unexpected, missing, unexpected


required = {"id", "name", "email"}

record = {
    "id": 10,
    "name": "Ada",
    "username": "ada",
}

valid, missing, unexpected = validate_exact_keys(record, required)

print("Valid:", valid)
print("Missing:", missing)
print("Unexpected:", unexpected)

assert valid is False
assert missing == {"email"}
assert unexpected == {"username"}


Valid: False
Missing: {'email'}
Unexpected: {'username'}


# Problem 30 — Validate a Minimum Schema

Now the record may contain extra keys, but it must include all required keys.

Implement the check with set-style comparison.


In [32]:
# Solution 30

def contains_required_keys(
    record: Mapping[str, Any],
    required: set[str],
) -> bool:
    return required <= record.keys()


record = {
    "id": 1,
    "name": "Grace",
    "email": "grace@example.com",
    "role": "admin",
}

assert contains_required_keys(
    record,
    {"id", "name", "email"},
)

print("Minimum schema satisfied.")


Minimum schema satisfied.


# Problem 31 — Create a Read-Only Live Mapping API

Write a class that owns a private dictionary but exposes a read-only **live** mapping to callers.

Callers must see future changes, but must not be able to assign through the exposed mapping.


In [33]:
# Solution 31

class LiveReadOnlyRegistry:
    def __init__(self) -> None:
        self._data: dict[str, Any] = {}
        self._readonly = MappingProxyType(self._data)

    @property
    def mapping(self):
        return self._readonly

    def register(self, name: str, value: Any) -> None:
        self._data[name] = value

    def unregister(self, name: str) -> None:
        del self._data[name]


registry = LiveReadOnlyRegistry()
public_mapping = registry.mapping

registry.register("alpha", 1)
registry.register("beta", 2)

print(public_mapping)

assert dict(public_mapping) == {"alpha": 1, "beta": 2}

try:
    public_mapping["gamma"] = 3
except TypeError as exc:
    print("Read-only as expected:", exc)


{'alpha': 1, 'beta': 2}
Read-only as expected: 'mappingproxy' object does not support item assignment


# Problem 32 — Benchmark View Creation vs Reusing a View

Compare these patterns:

```python
for _ in range(...):
    for key in d.keys():
        ...
```

and:

```python
keys = d.keys()
for _ in range(...):
    for key in keys:
        ...
```

The benchmark is educational only; avoid overgeneralizing tiny differences.


In [34]:
# Solution 32

d = {i: i for i in range(5_000)}
keys = d.keys()

def recreate_view() -> None:
    for _ in range(50):
        for key in d.keys():
            pass


def reuse_view() -> None:
    for _ in range(50):
        for key in keys:
            pass


t_recreate = timeit(recreate_view, number=100)
t_reuse = timeit(reuse_view, number=100)

print(f"Recreate view: {t_recreate:.4f}s")
print(f"Reuse view:    {t_reuse:.4f}s")


Recreate view: 0.5316s
Reuse view:    0.6618s


### Best practice

Do not cache a view merely for micro-optimization.

Cache it when it improves program structure or when you intentionally want a long-lived live view. Performance differences from creating a small view object are usually secondary to algorithmic design.


# Problem 33 — Debug a Function That Mutates Through an Alias

This code looks like it iterates over `current` and mutates `other`:

```python
current = data
other = data

for key in current:
    other.pop(key)
```

Explain why this is still unsafe and fix it.


In [35]:
# Solution 33

data = {"a": 1, "b": 2, "c": 3}

current = data
other = data

print("Same object:", current is other)
assert current is other

# Safe version: snapshot the keys first.
for key in list(current):
    other.pop(key)

print(data)

assert data == {}


Same object: True
{}


### Lesson

Mutation safety depends on object identity, not variable names.

Two different variables can reference the same dictionary.


# Problem 34 — Preserve Ordering While Filtering

Filter a dictionary to retain only records whose score is at least 70 while preserving the original insertion order.

Solve it by constructing a new dictionary.


In [36]:
# Solution 34

scores = {
    "Ada": 91,
    "Linus": 68,
    "Grace": 95,
    "Guido": 72,
    "Barbara": 64,
}

passed = {
    name: score
    for name, score in scores.items()
    if score >= 70
}

print(passed)

assert list(passed) == ["Ada", "Grace", "Guido"]


{'Ada': 91, 'Grace': 95, 'Guido': 72}


# Problem 35 — Streaming-Like Consumption vs Snapshot Processing

Implement two functions:

- `consume_lifo(d)` destructively consumes a dictionary with `popitem()`.
- `snapshot_process(d)` returns a list of the current items without changing the dictionary.

Demonstrate the semantic difference.


In [37]:
# Solution 35

def consume_lifo(d: dict[Any, Any]) -> list[tuple[Any, Any]]:
    result = []
    while d:
        result.append(d.popitem())
    return result


def snapshot_process(d: Mapping[Any, Any]) -> list[tuple[Any, Any]]:
    return list(d.items())


d1 = {"a": 1, "b": 2, "c": 3}
d2 = {"a": 1, "b": 2, "c": 3}

consumed = consume_lifo(d1)
snapshotted = snapshot_process(d2)

print("Consumed:", consumed)
print("d1 after:", d1)
print("Snapshot:", snapshotted)
print("d2 after:", d2)

assert d1 == {}
assert d2 == {"a": 1, "b": 2, "c": 3}


Consumed: [('c', 3), ('b', 2), ('a', 1)]
d1 after: {}
Snapshot: [('a', 1), ('b', 2), ('c', 3)]
d2 after: {'a': 1, 'b': 2, 'c': 3}


# Problem 36 — Update Nested Mutable Values While Iterating

The dictionary maps project names to mutable lists of tasks.

Append `"review"` to every task list while iterating over `.items()`.

Explain why this differs from adding or removing dictionary keys.


In [38]:
# Solution 36

projects = {
    "compiler": ["parse", "optimize"],
    "website": ["design", "deploy"],
}

for project, tasks in projects.items():
    tasks.append("review")

print(projects)

assert projects == {
    "compiler": ["parse", "optimize", "review"],
    "website": ["design", "deploy", "review"],
}


{'compiler': ['parse', 'optimize', 'review'], 'website': ['design', 'deploy', 'review']}


### Key distinction

The dictionary's structure did not change.

The mutable objects stored *as values* changed internally. Dictionary iteration tracks the mapping structure, not arbitrary mutations inside referenced value objects.


# Problem 37 — Selective Structural Mutation with a Snapshot of Keys Only

Suppose computing whether to delete a record requires looking up its current value at deletion time.

Why might `list(d)` be preferable to `list(d.items())`?

Demonstrate a case where values can change between snapshot creation and processing.


In [39]:
# Solution 37

d = {
    "a": 1,
    "b": 2,
    "c": 3,
}

keys_snapshot = list(d)

# Simulate values changing after the key snapshot.
d["b"] = 200

for key in keys_snapshot:
    if d[key] >= 100:
        del d[key]

print(d)

assert d == {"a": 1, "c": 3}


{'a': 1, 'c': 3}


### Design insight

A key-only snapshot stabilizes the traversal domain while allowing each iteration to read the latest value.

An item snapshot stabilizes both the key sequence and the value references captured at snapshot time.


# Problem 38 — Use a View for a Live Dashboard Counter

Implement a tiny dashboard object that keeps a value view and reports:

- current count of values,
- current sum,
- current average.

Mutate the original dictionary after dashboard creation and prove the metrics update.


In [40]:
# Solution 38

class MetricsDashboard:
    def __init__(self, metrics: dict[str, float]) -> None:
        self._values = metrics.values()

    def count(self) -> int:
        return len(self._values)

    def total(self) -> float:
        return sum(self._values)

    def average(self) -> float:
        if not self._values:
            return 0.0
        return self.total() / self.count()


metrics = {"cpu": 30.0, "memory": 50.0}
dashboard = MetricsDashboard(metrics)

assert dashboard.total() == 80.0

metrics["disk"] = 40.0
metrics["cpu"] = 20.0

print("Count:", dashboard.count())
print("Total:", dashboard.total())
print("Average:", dashboard.average())

assert dashboard.count() == 3
assert dashboard.total() == 110.0


Count: 3
Total: 110.0
Average: 36.666666666666664


# Problem 39 — Write a Defensive Delete Helper

Implement:

```python
delete_where(d, predicate)
```

It should delete every entry for which `predicate(key, value)` is true.

Requirements:

- no iterator invalidation,
- return the number of deleted entries,
- make only one pass to decide what must be deleted.


In [41]:
# Solution 39

def delete_where(
    d: dict[Any, Any],
    predicate: Callable[[Any, Any], bool],
) -> int:
    keys_to_delete = [
        key
        for key, value in d.items()
        if predicate(key, value)
    ]

    for key in keys_to_delete:
        del d[key]

    return len(keys_to_delete)


data = {
    "alpha": 1,
    "beta": 10,
    "gamma": 3,
    "delta": 20,
}

deleted = delete_where(
    data,
    lambda key, value: value >= 10,
)

print("Deleted:", deleted)
print("Remaining:", data)

assert deleted == 2
assert data == {"alpha": 1, "gamma": 3}


Deleted: 2
Remaining: {'alpha': 1, 'gamma': 3}


# Problem 40 — Advanced Challenge: Transactional Dictionary Transformation

Implement a transformation that may:

- rename keys,
- transform values,
- reject duplicate resulting keys.

The original dictionary must remain completely unchanged if any collision or transformation error occurs.

Only after the full transformed result is valid should the original dictionary be replaced.

This pattern is useful when structural changes must be **transactional**.


In [42]:
# Solution 40

def transform_mapping_transactionally(
    d: dict[Any, Any],
    key_transform: Callable[[Any], Any],
    value_transform: Callable[[Any, Any], Any],
) -> None:
    staged: dict[Any, Any] = {}

    for old_key, old_value in d.items():
        new_key = key_transform(old_key)
        new_value = value_transform(old_key, old_value)

        if new_key in staged:
            raise ValueError(
                f"duplicate transformed key: {new_key!r}"
            )

        staged[new_key] = new_value

    # Commit only after every transformation succeeded.
    d.clear()
    d.update(staged)


data = {
    "a": 10,
    "b": 20,
    "c": 30,
}

transform_mapping_transactionally(
    data,
    key_transform=str.upper,
    value_transform=lambda key, value: value * 2,
)

print(data)

assert data == {
    "A": 20,
    "B": 40,
    "C": 60,
}


{'A': 20, 'B': 40, 'C': 60}


In [43]:
# Transactional failure demonstration

data = {
    "A": 1,
    "a": 2,
}

before = data.copy()

try:
    transform_mapping_transactionally(
        data,
        key_transform=str.lower,
        value_transform=lambda key, value: value,
    )
except ValueError as exc:
    print("Expected failure:", exc)

print("Still unchanged:", data)

assert data == before


Expected failure: duplicate transformed key: 'a'
Still unchanged: {'A': 1, 'a': 2}


# Bonus Problem 41 — What Exactly Does a View Keep Live?

Test the following:

1. Create `items = d.items()`.
2. Replace an immutable value.
3. Mutate a mutable value in place.
4. Add a key.
5. Remove a key.

Observe the same view after every operation.


In [44]:
# Solution 41

d = {
    "number": 1,
    "list": [10, 20],
}

items = d.items()

print("Start:", items)

d["number"] = 999
print("After replacing immutable value:", items)

d["list"].append(30)
print("After mutating nested list:", items)

d["new"] = "added"
print("After adding key:", items)

del d["number"]
print("After removing key:", items)


Start: dict_items([('number', 1), ('list', [10, 20])])
After replacing immutable value: dict_items([('number', 999), ('list', [10, 20])])
After mutating nested list: dict_items([('number', 999), ('list', [10, 20, 30])])
After adding key: dict_items([('number', 999), ('list', [10, 20, 30]), ('new', 'added')])
After removing key: dict_items([('list', [10, 20, 30]), ('new', 'added')])


# Bonus Problem 42 — A Mini Test Suite

The following tests summarize the most important guarantees and safe patterns from this notebook.


In [45]:
# Solution 42

def run_view_tests() -> None:
    # Live key view
    d = {"a": 1}
    keys = d.keys()
    d["b"] = 2
    assert set(keys) == {"a", "b"}

    # Live item view
    items = d.items()
    d["a"] = 10
    assert ("a", 10) in items

    # Safe value replacement
    for key in d:
        d[key] *= 2
    assert d == {"a": 20, "b": 4}

    # Safe structural mutation via snapshot
    for key in list(d):
        if key == "a":
            del d[key]
    assert d == {"b": 4}

    # Key-view set operations
    x = {"a": 1, "b": 2}
    y = {"b": 9, "c": 3}
    assert x.keys() & y.keys() == {"b"}
    assert x.keys() | y.keys() == {"a", "b", "c"}

    # Exact item equality is independent of insertion order
    p = {"x": 1, "y": 2}
    q = {"y": 2, "x": 1}
    assert p.items() == q.items()

    print("All dictionary-view tests passed.")


run_view_tests()


All dictionary-view tests passed.


# Summary: Production Best Practices

## Prefer views when

- you want live access to a dictionary,
- you need set operations on keys,
- you need efficient iteration over `(key, value)` pairs,
- you intentionally want later dictionary changes to remain visible.

## Prefer snapshots when

- you need stable iteration while changing keys,
- you need historical state,
- you need a fixed batch,
- you want to decouple lifetime from the original dictionary.

## Mutation rule of thumb

### Usually safe during iteration

```python
for key, value in d.items():
    d[key] = transform(value)
```

The key domain stays unchanged.

### Unsafe structural pattern

```python
for key in d:
    del d[key]
```

### Safe structural pattern

```python
for key in list(d):
    del d[key]
```

or:

```python
d = {
    key: value
    for key, value in d.items()
    if keep(key, value)
}
```

## Iteration best practice

If both key and value are needed:

```python
for key, value in d.items():
    ...
```

instead of:

```python
for key in d:
    value = d[key]
```

## Key-set comparison

Use views directly:

```python
missing = required - record.keys()
extra = record.keys() - allowed
common = first.keys() & second.keys()
```

## Final principle

A dictionary view is a **live interface to mapping state**, not a frozen container.

The most important design question is therefore:

> Do I want this code to observe future dictionary changes, or do I want a stable snapshot?

Choosing deliberately between those two semantics prevents many subtle bugs.
